### Default hf dataloading

In [1]:
from transformers import default_data_collator
from transformers import (
    CONFIG_MAPPING,
    MODEL_FOR_CAUSAL_LM_MAPPING,
    AutoConfig,
    AutoModelForCausalLM,
    AutoTokenizer,
    HfArgumentParser,
    Trainer,
    TrainingArguments,
    default_data_collator,
    is_torch_xla_available,
    set_seed,
    get_scheduler,
)
import transformers
from itertools import chain

# from torchtune.models.llama3_2 import llama3_2_1b
from transformers.testing_utils import CaptureLogger
from torch.utils.data import DataLoader
from datasets import load_dataset

tok_logger = transformers.utils.logging.get_logger(
    "transformers.tokenization_utils_base"
)


raw_datasets = load_dataset(
    "HuggingFaceFW/fineweb-edu",
    name="sample-10BT",
    split="train",
    cache_dir="fineweb_edu_10b",
    num_proc=16,
)
raw_datasets = raw_datasets.select(list(range(len(raw_datasets) // 6)))

# column_names = list(raw_datasets["train"].features)
column_names = list(raw_datasets.features)
text_column_name = "text" if "text" in column_names else column_names[0]
block_size = 2048
model_name_or_path = "unsloth/Llama-3.2-1B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(
    model_name_or_path,
    use_fast=True,
)


def tokenize_function(examples):
    with CaptureLogger(tok_logger) as cl:
        output = tokenizer(examples[text_column_name])
    # clm input could be much much longer than block_size
    if "Token indices sequence length is longer than the" in cl.out:
        tok_logger.warning(
            "^^^^^^^^^^^^^^^^ Please ignore the warning above - this long input will be chunked into smaller bits"
            " before being passed to the model."
        )
    return output


tokenized_datasets = raw_datasets.map(
    tokenize_function,
    batched=True,
    num_proc=16,
    remove_columns=column_names,
    # load_from_cache_file=not data_args.overwrite_cache,
    desc="Running tokenizer on dataset",
)


def group_texts(examples):
    # Concatenate all texts.
    concatenated_examples = {k: list(chain(*examples[k])) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    # We drop the small remainder, and if the total_length < block_size  we exclude this batch and return an empty dict.
    # We could add padding if the model supported it instead of this drop, you can customize this part to your needs.
    total_length = (total_length // block_size) * block_size
    # Split by chunks of max_len.
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result


lm_datasets = tokenized_datasets.map(
    group_texts,
    batched=True,
    num_proc=16,
    desc=f"Grouping texts in chunks of {block_size}",
)
# lm_datasets = lm_datasets.remove_columns(
#     column_names=[item for item in lm_datasets.features.keys() if item != "input_ids"]
# )

/opt/conda/lib/python3.11/site-packages/torch/cuda/__init__.py:789: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
/opt/conda/lib/python3.11/site-packages/torch/cuda/__init__.py:789: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/98 [00:00<?, ?it/s]

In [2]:
lm_datasets

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 790587
})

In [ ]:
# train_dataset = lm_datasets["train"]
train_dataset = lm_datasets
# batch_size = 32 * 2
batch_size = 2 * 2 * 2 * 2 * 2 * 2
train_dataloader = DataLoader(
    train_dataset,
    shuffle=True,
    collate_fn=default_data_collator,
    batch_size=batch_size,
    drop_last=True,
    num_workers=16,
    # persistent_workers=True,
    pin_memory=True,
)

In [10]:
from tqdm import tqdm

for i in range(5):
    for item in tqdm(train_dataloader):
        item
# ~630 it\s batch 4
# ~380 it\s batch 16
# ~250 it\s batch 32
# ~150 it\s batch 64

 13%|█▎        | 1584/12352 [00:11<01:19, 135.49it/s]


KeyboardInterrupt: 

In [11]:
item

{'input_ids': tensor([[ 6913,  4967,   584,  ...,  7154,   220,    22],
         [  555,   220,  1049,  ...,   584,  7846,  1431],
         [13147,   627,   644,  ...,   810,  8162,   501],
         ...,
         [58054,   304,  9191,  ...,   627,    12, 31913],
         [  315, 84998,   690,  ...,   775,  1436,  1935],
         [46054,  5029,   311,  ..., 18618,   449,   279]]),
 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 1, 1],
         ...,
         [1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 1, 1]]),
 'labels': tensor([[ 6913,  4967,   584,  ...,  7154,   220,    22],
         [  555,   220,  1049,  ...,   584,  7846,  1431],
         [13147,   627,   644,  ...,   810,  8162,   501],
         ...,
         [58054,   304,  9191,  ...,   627,    12, 31913],
         [  315, 84998,   690,  ...,   775,  1436,  1935],
         [46054,  5029,   311,  ..., 18618,   449,   2

### Mosaic streaming dataloading

In [1]:
from streaming import MDSWriter, StreamingDataset
from torch.utils.data import DataLoader
from transformers import default_data_collator

# local_dir = "fineweb_edu_10b_numpy_mds_chunked"
# local_dir = "fineweb_edu_10b_numpy_mds_chunked_1024"
local_dir = "fineweb_edu_10b_numpy_mds_chunked_2048"
# local_dir = "fineweb_edu_10b_numpy_mds_chunked"
# batch_size = 16 * 2 * 2
# batch_size = 16
batch_size = 4
dataset = StreamingDataset(
    local=local_dir,
    remote=local_dir,
    batch_size=batch_size,
    # batch_size=1,
    # batch_size=64,
    split=None,
    shuffle=True,
)
dataloader = DataLoader(
    dataset,
    batch_size=batch_size,
    pin_memory=True,
    num_workers=4,
    collate_fn=default_data_collator,
    drop_last=True,
    # shuffle=True,
    # persistent_workers=True,
)

In [2]:
next(iter(dataset))['input_ids'].shape

(2048,)

In [2]:
next(iter(dataloader))['input_ids'].shape

torch.Size([4, 2048])

In [3]:
import torch
isinstance(dataset, torch.utils.data.IterableDataset)
isinstance(dataset, torch.utils.data.DataLoader)

False

In [4]:
isinstance(dataloader.dataset, torch.utils.data.IterableDataset)

True

In [ ]:
dataset[0]["attention_mask"].dtype

dtype('int64')

In [23]:
from tqdm import tqdm

for i in range(5):
    for item in tqdm(dataloader):
        item
# ~1134 it\s batch 4
# ~750 it\s batch 16
# ~550 it\s batch 32
# ~350 it\s batch 64

 24%|██▍       | 18698/78071 [00:54<02:54, 340.64it/s]

Unexpected exception formatting exception. Falling back to standard exception



Traceback (most recent call last):
  File "/opt/conda/lib/python3.11/site-packages/IPython/core/interactiveshell.py", line 3672, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_315578/2047797918.py", line 4, in <module>
    for item in tqdm(dataloader):
  File "/opt/conda/lib/python3.11/site-packages/tqdm/std.py", line 1181, in __iter__
    for obj in iterable:
  File "/opt/conda/lib/python3.11/site-packages/torch/utils/data/dataloader.py", line 733, in __next__
    data = self._next_data()
           ^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/torch/utils/data/dataloader.py", line 1491, in _next_data
    idx, data = self._get_data()
                ^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/torch/utils/data/dataloader.py", line 1453, in _get_data
    success, data = self._try_get_data()
                    ^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/torch/utils/data/dataloade

In [ ]:
item["input_ids"].shape

torch.Size([64, 2048])

In [25]:
next(iter(dataloader))

{'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 1, 1],
         ...,
         [1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 1, 1]]),
 'input_ids': tensor([[  315, 15203,   701,  ...,  6514,    13,  1556],
         [  323,   420,   374,  ..., 48635,   889,   753],
         [ 7245,  5808,   374,  ...,    25,   220,   605],
         ...,
         [78976, 49493,  6439,  ...,   791, 17420,  1101],
         [81586,    11,   369,  ..., 63488,    11,   449],
         [ 7006,  2191,   358,  ...,    11,  9405,   389]]),
 'labels': tensor([[  315, 15203,   701,  ...,  6514,    13,  1556],
         [  323,   420,   374,  ..., 48635,   889,   753],
         [ 7245,  5808,   374,  ...,    25,   220,   605],
         ...,
         [78976, 49493,  6439,  ...,   791, 17420,  1101],
         [81586,    11,   369,  ..., 63488,    11,   449],
         [ 7006,  2191,   358,  ...,    11,  9405,   3

In [26]:
next(iter(train_dataloader))

{'input_ids': tensor([[ 1101,  1893,   264,  ..., 46090,  1124,    13],
         [   11,   505,  1633,  ...,  1648,   311,  6420],
         [ 1047,  1027,  5496,  ...,  6873,  1887,    11],
         ...,
         [  527,   220,   975,  ...,  2753,  5644,   527],
         [  842,   315,  1855,  ..., 15813,  3777,   323],
         [43338,   420,  2362,  ...,  2200,  7829,   304]]),
 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 1, 1],
         ...,
         [1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 1, 1]]),
 'labels': tensor([[ 1101,  1893,   264,  ..., 46090,  1124,    13],
         [   11,   505,  1633,  ...,  1648,   311,  6420],
         [ 1047,  1027,  5496,  ...,  6873,  1887,    11],
         ...,
         [  527,   220,   975,  ...,  2753,  5644,   527],
         [  842,   315,  1855,  ..., 15813,  3777,   323],
         [43338,   420,  2362,  ...,  2200,  7829,   3

### Test RAM dataloader

In [30]:
from tqdm import tqdm

total_items = 500_000
ram_dataset = []
for i in tqdm(range(total_items)):
    ram_dataset.append(dataset[i])

100%|██████████| 500000/500000 [00:30<00:00, 16615.83it/s]


In [ ]:
from torch.utils.data import Dataset


class SimpleListDataset(Dataset):

    def __init__(self, ram_data):
        self.dataset = ram_data

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        return self.dataset[idx]


ram_pytorch_dataset = SimpleListDataset(
    ram_data=ram_dataset,
)
batch_size = 16 * 2 * 2
ram_dataloader = DataLoader(
    ram_pytorch_dataset,
    batch_size=batch_size,
    pin_memory=True,
    num_workers=16,
    collate_fn=default_data_collator,
    drop_last=True,
    shuffle=True,
)

In [ ]:
from tqdm import tqdm

for i in range(5):
    for item in tqdm(ram_dataloader):
        item

# 2500 it\s 4 batch
# 1700 it\s 16 batch
# 950 it\s 32 batch
# 680 it\s 64 batch

 38%|███▊      | 2974/7812 [00:08<00:13, 353.27it/s]


KeyboardInterrupt: 